# Lab 3: Codifying Data Business Processes


## Goal

See how Declarative Automation Bundles (DABs), formerly known as Databricks Asset Bundles, and LakeFlow Jobs turn ad-hoc data work into automated, version-controlled, observable business processes.

## What you'll do

1. Review the orchestration job in the LakeFlow Jobs UI
1. Review the pipeline through the Lakeflow Pipelines UI
   - Understand how the pipeline resource YAML defines what you just saw in the UI
1. Review the bundle structure (infrastructure as code) that created the assets
1. See how jobs can be controlled programmatically

---

In [0]:
%run ../utils/config

## Part 1: Review the orchestration job in the Lakeflow Pipelines UI
Lakeflow Jobs are an orchestration mechanism for automating most any task in Databricks.  Task types include Notebook, Pipeline, Python script, Python wheel, SQL, and flow control (conditional, for each) task types.

In this workshop, we will mainly use Pipeline and Notebook task types.

### The job's tasks

**In the UI:**
1. Navigate to **Jobs & Pipelines**
   - Here you will find all of the Jobs and Pipelines defined in your workspace along with their run statuses.
2. Find `financial_intelligence_job`
3. Click to open it, then click **Tasks** to see the task Directed Acyclic Graph (DAG):
   - **Task 1:** `run_pipeline` — triggers the Spark Declarative Pipeline
   - **Task 2:** `build_search_table` — builds a table for vector search indexing which we will use later
   - **Task 3:** `validate_quality` — asserts row counts and data completeness
   - **Task 4:** `notify_complete` — prints a completion summary
4. Select each task and observe the configuration settings for each. _Note that the job configurations in this job are read-only because the job is defined through a Declarative Automation Bundle (DAB) that we will examine later._

This `financial_intelligence_job` job automates the `full pipeline → search table → validate → notify` workflow.  It is configured for manual triggering, but could be triggered on a schedule or in response to events, such as file arrival.

<img src="../utils/_resources/JobDAG.png" alt="Job DAG"/>

- Do these tasks run in order, or in parallel?
- Will the `notify_complete` task always run, or does the `validate_quality` task need to run successfully first?

### The job's execution history
**In the UI:**
1. Click on the **Runs** tab to see the run history
   - This job has been run (at least) five times.
   - The lower section shows the run history as a table with most recent runs first.
   - The upper section depicts the run status and relative duration of each task.  The colored blocks are clickable, enabling you to drill into execution details.

-  Which run took the longest to complete?
-  Why did the `validate_quality` task fail on the first three runs?

<img src="../utils/_resources/JobRuns.jpg" alt="Job Runs"/>

## Part 2: Review the pipeline in the Lakeflow Pipelines UI

A Lakeflow Spark Declarative Pipeline enables you to declare the "what" of the end state you wish to achieve while leaving "how" details to the infrastructure.

### The pipeline's execution history
The first task in the `financial_intelligence_job` is this pipeline.

**In the UI:**
1. Navigate back to the **Jobs & Pipelines** list page
2. Click `financial_intelligence_pipeline`.  A Lakeflow Spark Declarative Pipeline is oriented around the artifacts (tables, views) that it generates, not tasks.  The view being shown is generated as part of a run.
   <img src="../utils/_resources/PipelineRuns.jpg" alt="Pipeline Runs" width="100%"/>
3. Choose the **earliest run** from the run list near the top
   - Since this was the very first run, we see that the `financial_docs_gold` and `company_ai_investment_summary` materialized views indicate a `Full recompute`.
   - How many PDF documents do you think were processed in this first run? 
     - _Hint: Each PDF processed will result in a new row in the bronze and silver streaming tables._
   - What does the yellow indicator under **Warnings** mean on the `financial_docs_silver` row mean?
4. Click the **3 met | 1 unmet** link under **Expectations**
4. Now click **All** to see the status of all Expectations
   - There are four Expectations defined on the silver table.  Three were met for all rows processed.  One was not met for 1 PDF file processed.  *More on how the expectations are defined in a moment...*

<img src=../utils/_resources/Expectations.png>

5. Choose the **latest run** (the run at the top of the list)
   - Note that the `company_ai_investment_summary` table is `Incremental` in this run, meaning the table was not rebuilt from scratch, but rather had additions, modifications, and deletions applied incrementally.

<img src=../utils/_resources/Incrementalization.png>

   - Why was `financial_docs_gold` still a `Full recompute`?
     - This one is tricky: any materialized view that calls AI functions (like `ai_classify`, `ai_extract`, etc.) will trigger a full recompute on every update because the engine can't guarantee incremental consistency for non-deterministic expressions.


### Observe data quality programatically
The `event_log()` SQL function enables you to explore the same data quality events using code.

The Spark Declarative Pipeline applied data quality expectations at the silver layer, checking if content, company, document type, and fiscal period were detected on each document.

Let's see what the pipeline logged.

In [0]:
# Query the pipeline event log for data quality results
# The event_log() function queries the pipeline's internal audit log
event_log = spark.sql(f"""
    SELECT
        timestamp,
        details:flow_progress.data_quality.dropped_records    AS dropped,
        details:flow_progress.data_quality.warned_records     AS warned,
        details:flow_progress.data_quality.expectations    AS expectations
    FROM event_log(TABLE({shared_schema}.financial_docs_silver))
    WHERE event_type = 'flow_progress'
      AND details:flow_progress.data_quality.expectations IS NOT NULL
    ORDER BY timestamp DESC
    LIMIT 50
""")
display(event_log)

In [0]:
# How many silver rows have a null fiscal period field? — Records were kept but flagged
result = spark.sql(f"""
    SELECT
        COUNT(*) AS total_documents,
        COUNT(fiscal_period) AS with_fiscal_period,
        COUNT(*) - COUNT(fiscal_period) AS missing_fiscal_period
    FROM {shared_schema}.financial_docs_silver
""").collect()[0]

print(f"Total documents:          {result['total_documents']:,}")
print(f"Fiscal Period extracted:  {result['with_fiscal_period']:,} ({result['with_fiscal_period']/result['total_documents']:.1%})")
print(f"Fiscal Period missing:    {result['missing_fiscal_period']:,}")

**Key insight:** The pipeline's data quality layer is declarative — you write
`@dp.expect("rule", "condition")` or `@dp.expect_or_drop("rule", "condition")` once in code, and Unity Catalog tracks compliance
automatically. No ad-hoc validation scripts, no silent failures.

---

## Part 3: Declarative Automation Bundles
Formerly known as Databricks Asset Bundles, facilitate the adoption of software engineering best practices, including source control, code review, testing, and continuous integration and delivery (CI/CD)

Declarative Automation Bundles include:
-  Infrastructure and workspace configurations
-  Source files
-  Definitions for Databricks resources (i.e. Lakeflow Jobs, Lakeflow Spark Declarative Pipelines, Dashboards, Model Serving endpoints, MLflow Experiments, MLflow registered models, and others)
-  Unit tests and integration tests


<img src="../utils/_resources/bundles-cicd.png">

Declarative Automation Bundles are typically developed in a local development environment such as VSCode or Cursor.  Databricks also supports [DABs in the Workspace](https://www.databricks.com/blog/announcing-databricks-asset-bundles-now-workspace) which enables you to build, edit and deploy using DABs from within a Databricks workspace.  We will use DABs in the Workspace in this workshop so we won't need a local dev environment.

### Review the bundle structure

A Declarative Automation Bundle or Databricks Asset Bundle (DAB) is infrastructure as code for Databricks resources: pipelines, jobs, models, and serving endpoints — all defined in YAML.  

- DABs are typically deployed via the Databricks CLI from a local IDE, such as VisualStudio Code or Cursor, or CI/CD automation such as GitHub Actions.
- DABs can also be managed and deployed from within a Databricks workspace (see [Collaborating on bundles in the workspace](https://docs.databricks.com/aws/en/dev-tools/bundles/workspace)), which is how we will work with them in this lab.

**In the UI:**
1. Navigate to <span style="color:cc0000;background-color:#e0e0e0;font-family:courier">../bundles/financial_pipelines/</span> in the **Workspace Navigator**
2. Open <span style="color:cc0000;background-color:#e0e0e0;font-family:courier">databricks.yml</span> — the root definition file - in a new browser tab.

### The bundle root `databricks.yml`

**Key concepts in this bundle:**

- **`variables`** — parameterize catalog and schema names so the same bundle deploys to dev and prod
- **`targets`** — define environments (`dev` with development mode, `prod` with production mode)
- **`include`** — reference the resource YAML files that define jobs and pipelines
- **`mode: development`** — in dev mode, resource names get a `[dev {username}]` prefix to avoid conflicts and enable parallel development

TODO: I have several deployment targets defined for testing workspaces.  I will remove those, keeping only one production target example.

### The job resource YAML `orchestration_job.yml`

1.  In the other browser tab, **open** `bundles/financial_pipelines/resources/orchestration_job.yml`

The job you saw in the UI (full pipeline → search table → validate → notify ) was deployed from this file

Notice:
- Tasks, their source files, their compute requirements, and their dependencies are defined in code
- Bundle resources can reference other resources.  For example, the job's pipeline task references the pipeline bundle resource with `${resources.pipelines.financial_intelligence_pipeline.id}`
- Jobs can pass parameters to the child tasks, making them portable and reusable 
<img src=../utils/_resources/JobTask.png>

### The pipeline resource YAML `financial_pipeline.yml`

1.  In the other browser tab, **open** `bundles/financial_pipelines/resources/financial_pipeline.yml`

The pipeline you saw in the UI (bronze → silver → gold) was deployed from this file

Notice:
- Resource files can reference variable values defined in the bundle (typically in the targets) with the syntax `${var.var_name}`
- `channel: PREVIEW` enables `ai_parse_document` and `ai_extract` in pipeline notebooks
- `serverless: true` means no cluster configuration — Databricks manages compute automatically
- The pipeline points to the three `*.py` files in the `src` folder tree that define the Bronze, Silver, and Gold tables
<img src=../utils/_resources/PipelineDef.png>

### The pipeline library Python file `silver_extract.py`

1.  In the other browser tab, open `bundles/financial_pipelines/src/financial_intelligence_pipeline/transformations/silver_extract.py`
1.  This Spark Declarative Pipeline source file declares the `financial_docs_silver` streaming table

Notice:
- The `@dp` decorator is used to define streaming tables, materialized views, and to define expectations (quality checks)
- The file defines *what* to do, not *how* to do it

Spark Declarative Pipelines provide:
-  Batch and streaming processing
-  Incremental processing
-  Source and sinks to a broad spectrum of systems
-  Automated data quality checks

<img src=../utils/_resources/PipelineCode.png>

### The bundle editor
Databricks provides an editor specifically for bundles When working with bundles in the workspace

In the other browser tab:
1. Click the **Deployments** icon (<img src=../utils/_resources/DeploymentIcon.png width=40px>) in the left nav bar
2. Observe:
   - The targets are available in a drop-down list
   - The resources defined in the bundle are visible in the list
   - The **Add** button provides an easy way to add more resources to the bundle, such as jobs, pipelines, and dashboards.

Due to time constraints, **_Do not_** deploy the bundle or run the resources

## The Continuous Integration / Continuous Deployment Workflow

Databricks Declarative Automation Bundles (DABs) work hand-in-hand with deployment automation tools such as Azure DevOps, GitHub Actions, or Jenkins.  In this workshop, we will use GitHub Actions as our primary example.

The CI/CD pipeline has two triggers:

**On Pull Request** (<span style="color:cc0000;background-color:#e0e0e0;font-family:courier">.github/workflows/ci.yml</span>):
```
push to PR branch → run pytest → databricks bundle validate
```
Prevents merging broken code from reaching the main branch.  Whenever a Pull Request is issued, this workflow
1. runs unit tests (<span style="color:cc0000;background-color:#e0e0e0;font-family:courier">pytest</span>)
1. validates the bundle (<span style="color:cc0000;background-color:#e0e0e0;font-family:courier">databricks bundle validate</span>)

**On Merge to Main** (<span style="color:cc0000;background-color:#e0e0e0;font-family:courier">.github/workflows/deploy.yml</span>):
```
merge to main → bundle deploy → bundle run retrain_job
```
Automatically deploys and retrains after every approved code change.  Whenever code is merged back into the main branch, this workflow
1. runs unit tests (<span style="color:cc0000;background-color:#e0e0e0;font-family:courier">pytest</span>)
1. validates the bundle (<span style="color:cc0000;background-color:#e0e0e0;font-family:courier">databricks bundle validate</span>)
1. deploys the bundle (<span style="color:cc0000;background-color:#e0e0e0;font-family:courier">databricks bundle deploy --target prod</span>)
1. runs the job defined by the bundle (<span style="color:cc0000;background-color:#e0e0e0;font-family:courier">databricks bundle run financial_intelligence_job -target prod</span>)

<img src="../utils/_resources/CICD process.png">

## Part 4: Working with jobs programmatically
Jobs and pipelines can be managed programmatically using the Databricks SDK.


In [0]:
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

# Find the job by name
jobs = list(w.jobs.list(name="financial_intelligence_job"))

if not jobs:
    print("⚠️  Job not found. The instructor may need to run the bundle deploy.")
else:
    job = jobs[0]
    print(f"✓ Found job: {job.settings.name}  (ID: {job.job_id})")

You would use the following SDK call to run the job:

```
run = w.jobs.run_now(job_id=job.job_id)
```
Due to time constraints, we will not run the job in this lab.  Rather, let's explore past runs programmatically.

In [0]:
# Retrieve the most recent job run results (without triggering a new run)
runs = list(w.jobs.list_runs(job_id=job.job_id, limit=1))

if not runs:
    print("⚠️  No runs found for this job.")
else:
    latest = runs[0]
    run_id = latest.run_id
    life_cycle = latest.state.life_cycle_state.value
    result = latest.state.result_state.value if latest.state.result_state else "IN PROGRESS"

    host = spark.conf.get("spark.databricks.workspaceUrl")
    run_url = f"https://{host}/#job/{job.job_id}/run/{run_id}"

    print(f"Latest run ID:  {run_id}")
    print(f"State:          {life_cycle}")
    print(f"Result:         {result}")
    print(f"Start time:     {latest.start_time}")
    print(f"\n→ View run: {run_url}")

    # Show task-level results
    if latest.tasks:
        print(f"\nTask results:")
        for task in latest.tasks:
            task_state = task.state.result_state.value if task.state.result_state else "PENDING"
            print(f"  {task.task_key:<25} {task_state}")

In [0]:
# Compare the most recent job runs side by side
from datetime import datetime

recent_runs = list(w.jobs.list_runs(job_id=job.job_id, limit=10))

if not recent_runs:
    print("No runs found.")
else:
    # Build summary rows
    rows = []
    for r in recent_runs:
        start = datetime.fromtimestamp(r.start_time / 1000) if r.start_time else None
        end = datetime.fromtimestamp(r.end_time / 1000) if r.end_time else None
        duration_s = (r.end_time - r.start_time) / 1000 if r.end_time and r.start_time else None
        duration_str = f"{int(duration_s // 60)}m {int(duration_s % 60)}s" if duration_s else "—"

        task_results = {}
        if r.tasks:
            for t in r.tasks:
                task_results[t.task_key] = t.state.result_state.value if t.state.result_state else "PENDING"

        rows.append({
            "run_id": r.run_id,
            "started": start.strftime("%Y-%m-%d %H:%M") if start else "—",
            "duration": duration_str,
            "result": r.state.result_state.value if r.state.result_state else "IN PROGRESS",
            **task_results
        })

    df = spark.createDataFrame(rows)
    display(df)

## Part 5: Confirm results

The job and pipeline have been incrementally run multiple times.  Check the size of each table managed by the pipeline.

In [0]:
# Final row counts — confirmation that the pipeline ran successfully
for table in ["financial_docs_bronze", "financial_docs_silver", "financial_docs_gold", "company_ai_investment_summary"]:
    count = spark.sql(
        f"SELECT COUNT(*) AS cnt FROM {shared_schema}.{table}"
    ).collect()[0]['cnt']
    print(f"  ✓ {table:<40} {count:>6,} rows")

## Key Takeaways

1. **Declarative Automation Bundles** define pipelines and jobs as code — version-controlled, reviewable, deployable to multiple environments
2. **Lakeflow Jobs** orchestrate multi-step workflows with task dependencies, quality gates, and failure notifications
3. **Infrastructure as code** means a new team member can clone this repo and deploy the entire platform in one command
4. **The full platform** — ingest → parse → extract → classify → validate → notify — is automated and observable without any manual steps

---

## Session 1 Complete

You've built the foundational data layer for an AI-powered financial intelligence platform:
- ✓ **Governed** with Unity Catalog (schemas, lineage, access control, quality)
- ✓ **Automated** with Spark Declarative Pipelines and LakeFlow Jobs
- ✓ **Codified** with Declarative Automation Bundles
- ✓ **AI-powered** with `ai_parse_document`, `ai_extract`, and `ai_classify`

---



**Next:** Session 2 — Semantic, graph, and contextual data layers for AI
